In [5]:
import gzip
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
import pickle
from sklearn.preprocessing import LabelEncoder

# --- 1. Setup and Data Loading ---
AUTH_FILE = "../data/auth.txt.gz"
COLS = [
    "RecordID", "User", "AccountName", "SourceComputer",
    "TargetComputer", "AuthPackage", "LogonType", "LogonAction", "Success"
]

def load_auth_data(nrows=500000):
    with gzip.open(AUTH_FILE, "rt") as f:
        df = pd.read_csv(f, sep=",", names=COLS, nrows=nrows, header=None)
    # Create a boolean mask for machine accounts
    df['IsMachine'] = df['User'].str.contains(r'\$|ANONYMOUS LOGON', case=False, na=False)
    return df

df = load_auth_data(nrows=500000)

# --- 2. Enhanced Feature Engineering ---
def get_features(df_slice):
    stats = df_slice.groupby('User').agg(
        total_logins=('RecordID', 'count'),
        failed_logins=('Success', lambda x: (x == 'Fail').sum()),
        unique_comps=('TargetComputer', 'nunique'),
        unique_types=('LogonType', 'nunique')
    )
    # Feature Engineering
    stats['fail_rate'] = stats['failed_logins'] / stats['total_logins']
    stats['comp_diversity'] = stats['unique_comps'] / stats['total_logins']
    # Log transform counts to reduce the "Power User" bias
    stats['log_logins'] = np.log1p(stats['total_logins'])
    return stats

# --- 3. Segmented Rolling Ensemble ---
WINDOW_SIZE = 20000
STEP_SIZE = 10000 
feature_cols = ['log_logins', 'fail_rate', 'comp_diversity', 'unique_types']

all_window_results = []

for start in range(0, len(df) - WINDOW_SIZE + 1, STEP_SIZE):
    window_raw = df.iloc[start : start + WINDOW_SIZE]
    
    # Process Humans and Machines separately
    for is_machine_group in [True, False]:
        sub_df = window_raw[window_raw['IsMachine'] == is_machine_group]
        if sub_df.empty: continue
            
        X = get_features(sub_df)
        if len(X) < 20: continue # Need enough samples to find outliers
            
        X_scaled = StandardScaler().fit_transform(X[feature_cols])
        
        # Ensemble Models
        iso = IsolationForest(contamination=0.03, random_state=42).fit_predict(X_scaled)
        lof = LocalOutlierFactor(n_neighbors=min(20, len(X)-1), contamination=0.03).fit_predict(X_scaled)
        svm = OneClassSVM(nu=0.03, kernel='rbf').fit_predict(X_scaled)
        
        # Scoring (Anomaly = -1)
        X['ensemble_score'] = ((iso == -1).astype(int) + 
                               (lof == -1).astype(int) + 
                               (svm == -1).astype(int))
        X['window_start'] = start
        X['IsMachineAccount'] = is_machine_group
        all_window_results.append(X[['ensemble_score', 'window_start', 'IsMachineAccount']])

rolling_results = pd.concat(all_window_results)

# --- 4. Persistence Filter & Final Triage ---
# We want users flagged by at least 2 models in multiple windows
persistent_mask = rolling_results['ensemble_score'] >= 2
final_counts = rolling_results[persistent_mask].groupby(['User', 'IsMachineAccount']).size().rename('window_count').reset_index()

# Separate results for SOC review
human_threats = final_counts[~final_counts['IsMachineAccount']].sort_values('window_count', ascending=False)
machine_threats = final_counts[final_counts['IsMachineAccount']].sort_values('window_count', ascending=False)

# --- 5. Output ---
print(f"--- SOC ANALYSIS REPORT ({len(df)} rows) ---")
print(f"Human Anomalies Identified: {len(human_threats)}")
print(human_threats.head(10))
print(f"\nMachine Anomalies Identified: {len(machine_threats)}")
print(machine_threats.head(10))

# Export for LSTM tier
high_priority_users = final_counts[final_counts['window_count'] >= 3]['User'].unique().tolist()
print(f"\n[!] {len(high_priority_users)} users queued for LSTM Tier analysis.")

--- SOC ANALYSIS REPORT (500000 rows) ---
Human Anomalies Identified: 75
             User  IsMachineAccount  window_count
505      U22@DOM1             False            34
508     U252@DOM1             False            29
535         U73@?             False            23
503     U199@DOM1             False            21
504        U207@?             False            21
502        U199@?             False            19
536      U73@DOM1             False            19
476  SYSTEM@C4169             False            17
491    U1702@DOM1             False            16
498       U1825@?             False            13

Machine Anomalies Identified: 476
                     User  IsMachineAccount  window_count
25            C1114$@DOM1              True            49
430            C599$@DOM1              True            49
5    ANONYMOUS LOGON@C586              True            49
131              C2096$@?              True            49
168           C2480$@DOM1              True         

In [6]:
# --- 1. Setup Encoders ---
cat_cols = ['AuthPackage', 'LogonType', 'LogonAction', 'Success']
encoders = {col: LabelEncoder().fit(df[col].astype(str).tolist() + ['UNKNOWN']) for col in cat_cols}

# --- 2. Define LSTM Sequence Parameters ---
SEQ_LEN = 20  # We will look at blocks of 20 events at a time

def prepare_lstm_data(df, suspects, seq_len=20):
    lstm_input_data = {}
    
    # Calculate a simple 'Global Pulse' (failures per 1000 records)
    # This gives the LSTM a hint about network stability
    df['is_fail'] = (df['Success'] == 'Fail').astype(int)
    global_pulse = df['is_fail'].rolling(window=1000).mean().fillna(0)
    
    for user in suspects:
        user_df = df[df['User'] == user].sort_values('RecordID').copy()
        
        # Encode categorical features
        encoded_features = []
        for col in cat_cols:
            encoded_features.append(encoders[col].transform(user_df[col].astype(str)))
        
        # Add the Global Pulse
        encoded_features.append(global_pulse.loc[user_df.index].values)
        
        # Stack into a matrix (Events x Features)
        matrix = np.column_stack(encoded_features)
        
        # BREAK INTO SEQUENCES
        # If user has 100 events and seq_len is 20, we get 5 sequences
        sequences = []
        for i in range(0, len(matrix) - seq_len + 1, seq_len):
            sequences.append(matrix[i : i + seq_len])
            
        if sequences:
            lstm_input_data[user] = np.array(sequences)
            
    return lstm_input_data

# --- 3. Run and Save ---
suspects = final_counts[final_counts['window_count'] >= 3]['User'].unique()
final_payload = {
    'data': prepare_lstm_data(df, suspects, SEQ_LEN),
    'encoders': encoders,
    'seq_len': SEQ_LEN
}

with open('lstm_final_input.pkl', 'wb') as f:
    pickle.dump(final_payload, f)

print(f"Prepared sequences for {len(final_payload['data'])} users.")

Prepared sequences for 101 users.


In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# --- 1. The Best-Practice Architecture: LSTM Autoencoder ---
class AuthAutoencoder(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super(AuthAutoencoder, self).__init__()
        # Encoder: Compresses the sequence
        self.encoder = nn.LSTM(input_dim, hidden_dim, num_layers=2, batch_first=True)
        # Decoder: Reconstructs the sequence
        self.decoder = nn.LSTM(hidden_dim, hidden_dim, num_layers=2, batch_first=True)
        self.output_layer = nn.Linear(hidden_dim, input_dim)

    def forward(self, x):
        # We only need the final hidden state (hn) from the encoder
        _, (hn, _) = self.encoder(x)
        
        # Take the hidden state from the last layer and repeat it for the sequence length
        # This acts as the "bottleneck" context
        latent_context = hn[-1].unsqueeze(1).repeat(1, x.shape[1], 1)
        
        # Decode back to original shape
        x_decoded, _ = self.decoder(latent_context)
        return self.output_layer(x_decoded)

# --- 2. Prepare Data ---
# Flatten the dictionary of users into a single training pool of sequences
all_sequences = np.concatenate(list(final_payload['data'].values()))
X_tensor = torch.FloatTensor(all_sequences)

# Autoencoders are "Self-Supervised": the target (y) is the same as the input (X)
train_loader = DataLoader(TensorDataset(X_tensor, X_tensor), batch_size=64, shuffle=True)

# --- 3. Training Logic ---
INPUT_DIM = 5  # [Pkg, Type, Action, Success, Pulse]
HIDDEN_DIM = 32 # Compressed representation size
model = AuthAutoencoder(INPUT_DIM, HIDDEN_DIM)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print(f"Training Autoencoder on {X_tensor.shape[0]} sequences...")

epochs = 30
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for batch_x, _ in train_loader:
        optimizer.zero_grad()
        reconstruction = model(batch_x)
        loss = criterion(reconstruction, batch_x)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{epochs} | Reconstruction Loss: {total_loss/len(train_loader):.6f}")

print("Training Complete.")

Training Autoencoder on 7827 sequences...
Epoch 5/30 | Reconstruction Loss: 1.270155
Epoch 10/30 | Reconstruction Loss: 0.918613
Epoch 15/30 | Reconstruction Loss: 0.860750
Epoch 20/30 | Reconstruction Loss: 0.782574
Epoch 25/30 | Reconstruction Loss: 0.726135
Epoch 30/30 | Reconstruction Loss: 0.681110
Training Complete.


In [23]:
model.eval()
anomaly_reports = []

with torch.no_grad():
    for user, sequences in final_payload['data'].items():
        seq_tensor = torch.FloatTensor(sequences)
        reconstruction = model(seq_tensor)
        
        # Calculate MSE for every single sequence
        # (Events, Seq_Len, Features) -> mean error per sequence
        mse_per_sequence = torch.mean((seq_tensor - reconstruction)**2, dim=(1, 2)).numpy()
        
        for i, error in enumerate(mse_per_sequence):
            anomaly_reports.append({
                'User': user,
                'Sequence_ID': i,
                'Reconstruction_Error': error
            })

# Convert to DataFrame
anomaly_df = pd.DataFrame(anomaly_reports)

# Calculate User-Level risk based on their "Worst" moment
user_triage = anomaly_df.groupby('User')['Reconstruction_Error'].agg(['max', 'mean', 'count']).sort_values('max', ascending=False)

# 1. Basic Counts
total_suspects = len(user_triage)
machine_accounts = user_triage[user_triage.index.str.contains('\$')].copy()
human_accounts = user_triage[~user_triage.index.str.contains('\$')].copy()

# 2. Proportions
prop_machines = (len(machine_accounts) / total_suspects) * 100
prop_humans = (len(human_accounts) / total_suspects) * 100

# 3. Aggregated 'Surprise' Scores
avg_machine_max = machine_accounts['max'].mean()
avg_human_max = human_accounts['max'].mean()

print(f"--- LSTM TIER SUMMARY ---")
print(f"Total Unique Users in Triage: {total_suspects}")
print(f"Machine Accounts ($): {len(machine_accounts)} ({prop_machines:.1f}%)")
print(f"Human Accounts:        {len(human_accounts)} ({prop_humans:.1f}%)")
print("-" * 30)
print(f"Avg 'Max Surprise' for Machines: {avg_machine_max:.4f}")
print(f"Avg 'Max Surprise' for Humans:   {avg_human_max:.4f}")

# Isolate human accounts (those without a '$' in the name)
human_stats = user_triage[~user_triage.index.str.contains('\$')].copy()

# Calculate Peer Z-Score specifically for the human group
human_stats['peer_z_score'] = (human_stats['max'] - human_stats['max'].mean()) / human_stats['max'].std()

print("--- TOP 5 ANOMALIES BY PEER Z-SCORE (HUMANS) ---")
print(human_stats[['max', 'peer_z_score']].sort_values('peer_z_score', ascending=False).head(5))

# Calculate Z-Scores for the Max Error
user_triage['z_score'] = (user_triage['max'] - user_triage['max'].mean()) / user_triage['max'].std()

# Separate Machines and Humans for peer-group comparison
machine_stats = user_triage[user_triage.index.str.contains('\$')].copy()
machine_stats['peer_z_score'] = (machine_stats['max'] - machine_stats['max'].mean()) / machine_stats['max'].std()

print("--- TOP 5 ANOMALIES BY PEER Z-SCORE (MACHINES) ---")
print(machine_stats[['max', 'peer_z_score']].sort_values('peer_z_score', ascending=False).head(5))

--- LSTM TIER SUMMARY ---
Total Unique Users in Triage: 101
Machine Accounts ($): 76 (75.2%)
Human Accounts:        25 (24.8%)
------------------------------
Avg 'Max Surprise' for Machines: 1.9014
Avg 'Max Surprise' for Humans:   2.0348
--- TOP 5 ANOMALIES BY PEER Z-SCORE (HUMANS) ---
                           max  peer_z_score
User                                        
U94@DOM1              5.629787      1.882230
U68@DOM1              4.477587      1.278964
ANONYMOUS LOGON@C612  4.250305      1.159964
ANONYMOUS LOGON@C457  4.208549      1.138102
ANONYMOUS LOGON@C467  4.201739      1.134536
--- TOP 5 ANOMALIES BY PEER Z-SCORE (MACHINES) ---
                  max  peer_z_score
User                               
C3131$@DOM1  5.397521      2.445042
C1527$@DOM1  5.148313      2.270758
C885$@DOM1   5.112407      2.245647
C2653$@DOM1  4.981960      2.154419
C3580$@DOM1  4.590953      1.880968


In [40]:
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, clear_output

def grunt_proof_dashboard(user_id, anomaly_df, raw_df, payload, user_triage):
    # 1. Prep Data
    user_anomalies = anomaly_df[anomaly_df['User'] == user_id].sort_values('Sequence_ID')
    top_row = user_anomalies.loc[user_anomalies['Reconstruction_Error'].idxmax()]
    top_seq_idx = int(top_row['Sequence_ID'])
    
    seq_len = payload['seq_len']
    user_raw = raw_df[raw_df['User'] == user_id].sort_values('RecordID')
    evidence_block = user_raw.iloc[top_seq_idx * seq_len : (top_seq_idx + 1) * seq_len].copy()

    # 2. Persona Stats
    user_avg = user_anomalies['Reconstruction_Error'].mean()
    global_human_avg = user_triage[~user_triage.index.str.contains('\$')]['mean'].mean()

    # --- PLOTTING BLOCK ---
    fig = plt.figure(figsize=(18, 12))
    grid = plt.GridSpec(3, 2, wspace=0.3, hspace=0.4)

    # Risk Timeline
    ax1 = fig.add_subplot(grid[0, 0])
    ax1.plot(user_anomalies['Sequence_ID'], user_anomalies['Reconstruction_Error'], color='tab:red', marker='o')
    ax1.axhline(y=user_avg, color='green', linestyle='--', label='User Mean')
    ax1.set_title("RISK TIMELINE", fontweight='bold')
    ax1.legend()

    # Target Concentration
    ax2 = fig.add_subplot(grid[0, 1])
    target_counts = evidence_block['TargetComputer'].value_counts()
    sns.barplot(x=target_counts.index, y=target_counts.values, ax=ax2, palette="viridis")
    ax2.set_title("TARGET CONCENTRATION", fontweight='bold')

    # Protocol Signature
    ax3 = fig.add_subplot(grid[1, 0])
    evidence_block['AuthPackage'].value_counts().plot(kind='pie', autopct='%1.1f%%', ax=ax3, colors=sns.color_palette("pastel"))
    ax3.set_title("PROTOCOL SIGNATURE", fontweight='bold')
    ax3.set_ylabel("")

    # Success/Fail
    ax4 = fig.add_subplot(grid[1, 1])
    sns.countplot(data=evidence_block, x='Success', ax=ax4, palette={'Success': 'skyblue', 'Fail': 'salmon'})
    ax4.set_title("SUCCESS VS FAIL", fontweight='bold')

    # Persona Analysis
    ax5 = fig.add_subplot(grid[2, :])
    ax5.plot(user_anomalies['Sequence_ID'], user_anomalies['Reconstruction_Error'], color='blue', alpha=0.5)
    ax5.axhline(y=user_avg, color='green', linestyle='--', label=f'User Avg ({user_avg:.2f})')
    ax5.axhline(y=global_human_avg, color='red', linestyle=':', label=f'Peer Avg ({global_human_avg:.2f})')
    ax5.set_title("LIFETIME PERSONA ANALYSIS", fontweight='bold')
    ax5.legend(loc='upper right')

    plt.suptitle(f"EMERGENCY TRIAGE REPORT: {user_id}", fontsize=22, fontweight='bold', color='darkred')
    plt.show()

    # --- THE FEATURE ADDITION: EVIDENCE TABLE ---
    print(f"\n📑 RAW FORENSIC EVIDENCE: {user_id} | SEQUENCE ID: {top_seq_idx}")
    print("-" * 80)
    # Displaying the raw logs directly in the output
    display(evidence_block[['RecordID', 'SourceComputer', 'TargetComputer', 'AuthPackage', 'LogonType', 'Success']])

# --- RE-INITIALIZING THE WORKBENCH TO USE THE NEW DASHBOARD ---

def professional_soc_workbench(anomaly_df, raw_df, payload, user_triage):
    suspects = anomaly_df.groupby('User')['Reconstruction_Error'].max().sort_values(ascending=False).index.tolist()
    
    user_selector = widgets.Dropdown(options=suspects, description='🔍 Suspect:', layout={'width': '400px'})
    notes_box = widgets.Textarea(placeholder='Explain your decision...', description='📝 Notes:', layout={'width': '400px', 'height': '80px'})
    
    btn_escalate = widgets.Button(description='CRITICAL: Escalate Now', button_style='danger', icon='fire')
    btn_review = widgets.Button(description='Review Soon', button_style='warning', icon='clock')
    btn_ignore = widgets.Button(description='Mislabeled / False Positive', button_style='info', icon='check')
    
    output_area = widgets.Output()

    def record_decision(b):
        user = user_selector.value
        triage_results[user] = {
            'User': user, 'Category': b.description,
            'Max_Error': anomaly_df[anomaly_df['User'] == user]['Reconstruction_Error'].max(),
            'Notes': notes_box.value
        }
        with output_area:
            print(f"✅ RECORDED: {user} -> {b.description}")
            notes_box.value = ""

    for btn in [btn_escalate, btn_review, btn_ignore]:
        btn.on_click(record_decision)

    def on_user_change(change):
        if change['type'] == 'change' and change['name'] == 'value':
            with output_area:
                clear_output(wait=True)
                grunt_proof_dashboard(change['new'], anomaly_df, raw_df, payload, user_triage)

    user_selector.observe(on_user_change)
    
    # Layout with buttons at the top, then the dashboard
    display(widgets.VBox([user_selector, notes_box, widgets.HBox([btn_escalate, btn_review, btn_ignore])]))
    display(output_area)

    # Initial load
    with output_area:
        grunt_proof_dashboard(suspects[0], anomaly_df, raw_df, payload, user_triage)

# Run the final integrated tool
professional_soc_workbench(anomaly_df, df, final_payload, user_triage)

Output()

In [43]:
# 1. Get the RecordID of that C1710 Cleartext event
leak_record_id = df[(df['TargetComputer'] == 'C1710') & (df['LogonType'] == 'NetworkCleartext')]['RecordID'].min()

# 2. Look at all anomalies that happened BEFORE this RecordID
pre_leak_anomalies = anomaly_df[anomaly_df['User'].isin(human_accounts.index) & (anomaly_df['Sequence_ID'] * SEQ_LEN < leak_record_id)]

# 3. Find the SourceComputers these "early" anomalous users were coming FROM
early_sources = df[df['User'].isin(pre_leak_anomalies['User']) & (df['RecordID'] < leak_record_id)]['SourceComputer'].value_counts()

print("--- POTENTIAL PATIENT ZERO HOSTS ---")
print(early_sources.head(5))

Users seen acting from the suspect 'Patient Zero' host:
User
C1727$@DOM1     229
U66@DOM1        108
U78@DOM1       1356
dtype: int64


In [44]:
# Did an anomalous user log into a machine, and then a 
# DIFFERENT user started acting weird from that same machine?
cross_contamination = df[df['SourceComputer'] == 'C1727'].groupby('User').size()
print("Users seen acting from the suspect 'Patient Zero' host:")
print(cross_contamination)

Users seen acting from the suspect 'Patient Zero' host:
User
C1727$@DOM1     229
U66@DOM1        108
U78@DOM1       1356
dtype: int64


In [45]:
# Check if U66 and U78 are hitting the same servers from C1727
u66_targets = set(df[(df['SourceComputer'] == 'C1727') & (df['User'] == 'U66@DOM1')]['TargetComputer'])
u78_targets = set(df[(df['SourceComputer'] == 'C1727') & (df['User'] == 'U78@DOM1')]['TargetComputer'])

shared_targets = u66_targets.intersection(u78_targets)
print(f"Shared Targets being attacked from C1727: {shared_targets}")

Shared Targets being attacked from C1727: {'C1727'}


In [46]:
print(df[df['User'] == 'U78@DOM1']['LogonType'].value_counts())

LogonType
?              1103
Network         569
Batch           174
Interactive      20
Name: count, dtype: int64


In [47]:
print(df[df['User'] == 'U78@DOM1']['AuthPackage'].value_counts())

AuthPackage
?            1473
Kerberos      289
Negotiate     100
NTLM            4
Name: count, dtype: int64


In [48]:
anonymous_spikes = df[(df['TargetComputer'] == 'C1727') & (df['User'].str.contains('ANONYMOUS'))]
print(anonymous_spikes)

Empty DataFrame
Columns: [RecordID, User, AccountName, SourceComputer, TargetComputer, AuthPackage, LogonType, LogonAction, Success, IsMachine, is_fail]
Index: []


In [49]:
user_records = df[df['User'] == 'U78@DOM1']['RecordID']
print(f"Total Span (Records): {user_records.max() - user_records.min()}")
print(f"Events per 1000 logs: {len(user_records) / ((user_records.max() - user_records.min()) / 1000):.2f}")

Total Span (Records): 5113
Events per 1000 logs: 364.95


In [50]:
# Did U78 ever exist in the first 100,000 records?
early_existence = df[df['RecordID'] < 100000]['User'].unique()
is_new = 'U78@DOM1' not in early_existence
print(f"Is U78 a 'New' user appearing for the first time? {is_new}")


Is U78 a 'New' user appearing for the first time? False


In [55]:
# 1. Identify "Infrastructure" Nodes (Top 0.5% by In-Degree)
in_degree = df.groupby('TargetComputer')['User'].nunique()
infra_threshold = in_degree.quantile(0.995)
infrastructure_nodes = in_degree[in_degree >= infra_threshold].index.tolist()

results = []
# Loop through the suspects identified by the LSTM (the 101 users)
for user in suspects:
    user_df = df[df['User'] == user].sort_values('RecordID')
    
    # Calculate Velocity: Events per Record Span
    span = user_df['RecordID'].max() - user_df['RecordID'].min()
    count = len(user_df)
    velocity = count / span if span > 0 else 0
    
    # Calculate Entropy: Fraction of '?' or 'Negotiate' (Masked protocols)
    entropy_count = user_df['AuthPackage'].isin(['?', 'Negotiate']).sum()
    entropy_ratio = entropy_count / count
    
    # Check for Infrastructure Targeting
    hits_infra = user_df['TargetComputer'].isin(infrastructure_nodes).any()
    unique_targets = user_df['TargetComputer'].nunique()
    
    results.append({
        'User': user,
        'EventCount': count,
        'Velocity': velocity,
        'Entropy': entropy_ratio,
        'UniqueTargets': unique_targets,
        'Hits_Infra': hits_infra
    })

triage_summary = pd.DataFrame(results).sort_values('Velocity', ascending=False)

print("--- TOP 10 HIGH-VELOCITY THREATS (BOTS) ---")
print(triage_summary.head(10))

--- TOP 10 HIGH-VELOCITY THREATS (BOTS) ---
                     User  EventCount  Velocity   Entropy  UniqueTargets  \
97               U22@DOM1       13750  2.687647  0.538764             12   
103              U66@DOM1       10361  2.026007  0.528520            121   
75             C599$@DOM1        9605  1.877443  0.600521             21   
3    ANONYMOUS LOGON@C586        8878  1.735001  0.334647              1   
9             C1114$@DOM1        7583  1.482792  0.613741             21   
6              C104$@DOM1        6840  1.337243  0.601608             21   
72             C585$@DOM1        6766  1.324070  0.499852              2   
81             C743$@DOM1        5867  1.149265  0.501449              9   
71             C567$@DOM1        5181  1.012507  0.525188             21   
11             C123$@DOM1        4990  0.975371  0.530060             21   

     Hits_Infra  
97         True  
103       False  
75         True  
3          True  
9          True  
6          

In [56]:
# Check if the top high-velocity users are coming from C1727
top_bots = triage_summary.head(5)['User'].tolist()
bot_origins = df[df['User'].isin(top_bots)]['SourceComputer'].value_counts()

print("--- ORIGIN OF THE BOTS ---")
print(bot_origins.head(5))

--- ORIGIN OF THE BOTS ---
SourceComputer
C1619    5766
C1115    4564
C586     4382
C506     2229
C101     1890
Name: count, dtype: int64


In [57]:
# Identify the "Hot Path"
top_sources = ['C1619', 'C1115', 'C586']
pivots = df[df['SourceComputer'].isin(top_sources)].groupby(['SourceComputer', 'TargetComputer', 'User']).size().reset_index(name='LogonCount')

# Filter for logons that aren't self-loops
pivots = pivots[pivots['SourceComputer'] != pivots['TargetComputer']]
print("--- ACTIVE PIVOT PATHS ---")
print(pivots.sort_values('LogonCount', ascending=False).head(10))

--- ACTIVE PIVOT PATHS ---
    SourceComputer TargetComputer         User  LogonCount
141          C1619           C599   C599$@DOM1        1823
3            C1115          C1114  C1114$@DOM1        1499
86           C1619           C101   C599$@DOM1        1040
138          C1619           C553   C599$@DOM1         920
0            C1115           C101  C1114$@DOM1         839
180          C1619           C988   C599$@DOM1         580
84           C1115           C988  C1114$@DOM1         451
74           C1115           C523  C1114$@DOM1         412
89           C1619          C1085   C599$@DOM1         393
133          C1619           C523   C599$@DOM1         349


In [58]:
# Check the 'Success' rate for our Top 2 Bots
bot_users = ['U22@DOM1', 'U66@DOM1']
success_stats = df[df['User'].isin(bot_users)].groupby(['User', 'Success']).size().unstack(fill_value=0)
print("\n--- BOT SUCCESS/FAILURE RATIO ---")
print(success_stats)


--- BOT SUCCESS/FAILURE RATIO ---
Success   Fail  Success
User                   
U22@DOM1   849    12901
U66@DOM1     0    10361


In [59]:
# Find Machine Accounts hitting non-DC targets
machine_bots = triage_summary[triage_summary['User'].str.contains('\$')]['User'].head(5).tolist()
strange_machine_movement = df[df['User'].isin(machine_bots) & (~df['TargetComputer'].isin(['C586', 'C529']))]

print("\n--- ABNORMAL MACHINE ACCOUNT MOVEMENT ---")
print(strange_machine_movement[['User', 'SourceComputer', 'TargetComputer']].drop_duplicates())


--- ABNORMAL MACHINE ACCOUNT MOVEMENT ---
               User SourceComputer TargetComputer
85       C599$@DOM1           C553           C553
255      C104$@DOM1           C105           C523
1338     C599$@DOM1          C1065          C1065
1339     C599$@DOM1          C1619           C553
1663    C1114$@DOM1          C1115           C101
...             ...            ...            ...
224121   C743$@DOM1           C743           C457
225652   C743$@DOM1           C528           C528
225654   C743$@DOM1           C743           C528
227066   C743$@DOM1           C743          C1065
239844   C743$@DOM1          C1065          C1065

[107 rows x 3 columns]


In [60]:
# 1. Check the time density for U66
u66_data = df[df['User'] == 'U66@DOM1']
first_logon = u66_data['RecordID'].min()
last_logon = u66_data['RecordID'].max()
span = last_logon - first_logon

print(f"--- U66 ATTACK DENSITY ---")
print(f"Total Logons: {len(u66_data)}")
print(f"RecordID Span: {span}")
print(f"Avg Logons per Record: {len(u66_data) / span:.2f}")

# 2. Check the Entropy (The '?' protocol) specifically for U66's successes
print(f"\n--- U66 SUCCESS PROTOCOLS ---")
print(u66_data['AuthPackage'].value_counts())

--- U66 ATTACK DENSITY ---
Total Logons: 10361
RecordID Span: 5114
Avg Logons per Record: 2.03

--- U66 SUCCESS PROTOCOLS ---
AuthPackage
?           5476
Kerberos    4544
NTLM         341
Name: count, dtype: int64


In [61]:
# Where was the U66 Cannon firing from?
u66_origins = df[df['User'] == 'U66@DOM1']['SourceComputer'].value_counts()
print("--- THE U66 CANNON PLACEMENT ---")
print(u66_origins.head(5))

--- THE U66 CANNON PLACEMENT ---
SourceComputer
C1823    1105
C1697     368
C1952     310
C1971     251
C1732     245
Name: count, dtype: int64


In [62]:
# Did U22 also use these same "Cannon" hosts?
cannon_hosts = ['C1823', 'C1697', 'C1952', 'C1971', 'C1732']
co_conspirator = df[(df['SourceComputer'].isin(cannon_hosts)) & (df['User'] == 'U22@DOM1')]

print(f"--- CO-CONSPIRATOR ACTIVITY ON CANNONS ---")
if not co_conspirator.empty:
    print(co_conspirator['SourceComputer'].value_counts())
else:
    print("U22 was NOT on these cannons. They are separate attack threads.")


--- CO-CONSPIRATOR ACTIVITY ON CANNONS ---
U22 was NOT on these cannons. They are separate attack threads.


In [63]:
# Machines from both attack threads
thread_a_sources = ['C1823', 'C1697', 'C1952', 'C1971', 'C1732'] # U66 Cannons
thread_b_sources = ['C1619', 'C1115', 'C586', 'C506', 'C101']    # U22/Bot Hubs

# Who has logged into BOTH sets?
users_a = set(df[df['TargetComputer'].isin(thread_a_sources)]['User'])
users_b = set(df[df['TargetComputer'].isin(thread_b_sources)]['User'])

bridge_users = users_a.intersection(users_b)
print(f"--- THE BRIDGE USERS (The Possible Attacker Origin) ---")
print(bridge_users)

--- THE BRIDGE USERS (The Possible Attacker Origin) ---
{'U750@DOM1', 'C1823$@DOM1', 'U151@DOM1', 'U24@DOM1', 'C1971$@DOM1', 'C1732$@DOM1', 'U23@DOM1', 'U207@DOM1'}


In [64]:
# Rank bridge users by their first appearance (RecordID)
bridge_list = ['U750@DOM1', 'U151@DOM1', 'U24@DOM1', 'U23@DOM1', 'U207@DOM1']
first_appearances = df[df['User'].isin(bridge_list)].groupby('User')['RecordID'].min().sort_values()

print("--- TIMELINE OF THE BRIDGE USERS ---")
print(first_appearances)

--- TIMELINE OF THE BRIDGE USERS ---
User
U23@DOM1        1
U207@DOM1       2
U24@DOM1        2
U151@DOM1     203
U750@DOM1    2390
Name: RecordID, dtype: int64


In [65]:
# Where did the very first event (RecordID 1) occur?
genesis_event = df[df['RecordID'] == 1]
print("--- THE GENESIS OF THE ATTACK ---")
print(genesis_event[['User', 'SourceComputer', 'TargetComputer']])

--- THE GENESIS OF THE ATTACK ---
                     User SourceComputer TargetComputer
0    ANONYMOUS LOGON@C586          C1250           C586
1    ANONYMOUS LOGON@C586           C586           C586
2              C101$@DOM1           C988           C988
3             C1020$@DOM1          C1020          C1020
4             C1021$@DOM1          C1021           C625
..                    ...            ...            ...
239              U90@DOM1          C1786          C1786
240              U90@DOM1          C1786          C1786
241              U90@DOM1           C457           C457
242              U91@DOM1          C1787          C1787
243              U91@DOM1          C1787          C1787

[244 rows x 3 columns]


In [66]:
# --- THE RECONNAISSANCE FILTER ---
# We look for "UniqueTargets" > 100 with 100% success.
# Why: Humans hit 2-5 machines. Bots hit the whole subnet. 
# 100% success means they aren't guessing; they have the "Golden Ticket."
recon_check = triage_summary[triage_summary['UniqueTargets'] > 100]

# --- THE PROTOCOL ENTROPY CHECK ---
# We calculate the ratio of '?' in the AuthPackage column.
# Why: Windows logs '?' when a session is injected directly into memory (LSASS)
# bypassing the standard interactive login. This is the "Signature of Injection."
entropy_check = u66_data['AuthPackage'].value_counts(normalize=True)

# --- THE VELOCITY VERDICT ---
# Events / RecordID Span.
# Why: If Velocity > 1.0, the "user" is logging into machines faster 
# than the network is generating other logs. This is "Machine Speed."
velocity_val = len(u66_data) / (u66_data['RecordID'].max() - u66_data['RecordID'].min())

In [67]:
# 1. See which users pass the Reconnaissance Filter
print("--- USERS HITTING > 100 UNIQUE TARGETS ---")
print(triage_summary[triage_summary['UniqueTargets'] > 100])

# 2. See the Entropy (Injection Signature) for U66
print("\n--- U66 PROTOCOL ENTROPY (Injection Ratio) ---")
u66_data = df[df['User'] == 'U66@DOM1']
entropy_val = u66_data['AuthPackage'].value_counts(normalize=True).get('?', 0)
print(f"Percentage of 'Unknown' logons: {entropy_val * 100:.2f}%")

# 3. The Velocity Verdict
span = u66_data['RecordID'].max() - u66_data['RecordID'].min()
velocity_val = len(u66_data) / span if span > 0 else 0
print(f"\n--- U66 VELOCITY VERDICT ---")
print(f"Logons per Record: {velocity_val:.4f}")

--- USERS HITTING > 100 UNIQUE TARGETS ---
         User  EventCount  Velocity  Entropy  UniqueTargets  Hits_Infra
103  U66@DOM1       10361  2.026007  0.52852            121       False

--- U66 PROTOCOL ENTROPY (Injection Ratio) ---
Percentage of 'Unknown' logons: 52.85%

--- U66 VELOCITY VERDICT ---
Logons per Record: 2.0260


In [68]:
# Check if our Bridge Users ever touched the suspicious C1250
bridge_users = ['U750@DOM1', 'U151@DOM1', 'U24@DOM1', 'U23@DOM1']
c1250_audit = df[(df['TargetComputer'] == 'C1250') & (df['User'].isin(bridge_users))]

print("--- BRIDGE USERS LOGGING INTO C1250 ---")
print(c1250_audit)

--- BRIDGE USERS LOGGING INTO C1250 ---
Empty DataFrame
Columns: [RecordID, User, AccountName, SourceComputer, TargetComputer, AuthPackage, LogonType, LogonAction, Success, IsMachine, is_fail]
Index: []


In [69]:
# Look at the very first 500 records involving C1250
c1250_early = df[df['SourceComputer'] == 'C1250'].head(20)
print("--- C1250 PRE-ATTACK BEHAVIOR ---")
print(c1250_early[['RecordID', 'User', 'AuthPackage', 'Success']])

--- C1250 PRE-ATTACK BEHAVIOR ---
       RecordID                  User AuthPackage  Success
0             1  ANONYMOUS LOGON@C586        NTLM  Success
17            1           C1250$@DOM1    Kerberos  Success
1704          4           C1250$@DOM1   Negotiate  Success
75539       720  ANONYMOUS LOGON@C586        NTLM  Success
75550       720           C1250$@DOM1    Kerberos  Success
79399       758  ANONYMOUS LOGON@C586        NTLM  Success
79415       758           C1250$@DOM1    Kerberos  Success
89228       857           C1250$@DOM1    Kerberos  Success
89294       858           C1250$@DOM1    Kerberos  Success
89295       858           C1250$@DOM1           ?  Success
89296       858           C1250$@DOM1           ?  Success
89297       858           C1250$@DOM1    Kerberos  Success
89298       858           C1250$@DOM1    Kerberos  Success
89299       858           C1250$@DOM1    Kerberos  Success
89384       859           C1250$@DOM1        NTLM  Success
89469       860       

In [72]:
# Did anyone else attempt an ANONYMOUS LOGON at the start?
anon_origins = df[(df['User'].str.contains('ANONYMOUS')) & (df['RecordID'] < 10)]
print("--- OTHER POTENTIAL STARTING POINTS ---")
print(anon_origins['SourceComputer'].unique())

--- OTHER POTENTIAL STARTING POINTS ---
['C1250' 'C586' 'C1529' 'C4615' 'C2734' 'C3896' 'C671' 'C1065' 'C4622'
 'C213' 'C528' 'C1909' 'C457' 'C1186' 'C953' 'C1715' 'C3778' 'C493' 'C585'
 'C5336']


In [73]:
# We want to see every different user that touched C1250 in the first 1000 records
pivot_check = df[df['SourceComputer'] == 'C1250'].sort_values('RecordID')
print("--- THE BATON PASS ON C1250 ---")
print(pivot_check[['RecordID', 'User', 'Success']].drop_duplicates())

--- THE BATON PASS ON C1250 ---
        RecordID                  User  Success
0              1  ANONYMOUS LOGON@C586  Success
17             1           C1250$@DOM1  Success
1704           4           C1250$@DOM1  Success
75539        720  ANONYMOUS LOGON@C586  Success
75550        720           C1250$@DOM1  Success
79399        758  ANONYMOUS LOGON@C586  Success
79415        758           C1250$@DOM1  Success
89228        857           C1250$@DOM1  Success
89299        858           C1250$@DOM1  Success
89384        859           C1250$@DOM1  Success
89469        860           C1250$@DOM1  Success
91524        881           C1250$@DOM1  Success
92336        889           C1250$@DOM1  Success
92448        890           C1250$@DOM1  Success
92558        891           C1250$@DOM1  Success
92665        892           C1250$@DOM1  Success
145229      1442           C1250$@DOM1  Success
145219      1442  ANONYMOUS LOGON@C586  Success
178318      1781           C1250$@DOM1  Success
181761  

In [74]:
# PROOF OF LINK: 
# Show that when C1250 hits the DC, 
# other machines suddenly start succeeding with U66.

# 1. Get timestamps/RecordIDs of C1250's Anonymous hits
attack_windows = df[df['SourceComputer'] == 'C1250']['RecordID'].unique()

# 2. See what U66 was doing during those exact same RecordIDs
coordinated_attack = df[(df['User'] == 'U66@DOM1') & (df['RecordID'].isin(attack_windows))]

print("--- COORDINATED ACTION: U66 ACTIVITY DURING C1250 EXPLOIT WINDOWS ---")
print(coordinated_attack)

--- COORDINATED ACTION: U66 ACTIVITY DURING C1250 EXPLOIT WINDOWS ---
        RecordID      User  AccountName SourceComputer TargetComputer  \
187            1  U66@DOM1     U66@DOM1          C3868          C3868   
188            1  U66@DOM1     U66@DOM1          C3868          C3868   
75638        720  U66@DOM1     U66@DOM1          C1149          C1149   
75639        720  U66@DOM1     U66@DOM1          C1149          C1149   
89367        858  U66@DOM1     U66@DOM1          C2084          C2084   
89368        858  U66@DOM1     U66@DOM1          C2463          C2463   
89369        858  U66@DOM1     U66@DOM1           C779           C779   
89459        859  U66@DOM1     U66@DOM1          C2545          C2545   
89460        859  U66@DOM1     U66@DOM1          C2545          C2545   
89540        860  U66@DOM1     U66@DOM1          C1149          C1149   
89541        860  U66@DOM1     U66@DOM1          C2127          C2127   
89542        860  U66@DOM1     U66@DOM1          C2127

# Narrative:

Nothing actually suspicious in the first 500,000 logs. U22 and U66 had higher velocity than expected but could be startup scripts. Gemini thought this was a zero day exploit, hence the frantic 'proof' of ... nothing. But, it informs my understanding of what an exploit should look like in the auth dataset.